In [1]:
from pathlib import Path
import pandas as pd

print("Pandas版本：", pd.__version__)
print("当前工作目录：", Path.cwd())

Pandas版本： 3.0.3
当前工作目录： /Users/nyx/Desktop/user-behavior-analysis/notebooks


In [2]:
DATA_PATH = Path("../data/raw/user_behavior_processed.csv")

print("数据文件是否存在：", DATA_PATH.exists())
print("数据文件完整路径：", DATA_PATH.resolve())
print("文件大小：", round(DATA_PATH.stat().st_size / 1024**2, 2), "MB")

数据文件是否存在： True
数据文件完整路径： /Users/nyx/Desktop/user-behavior-analysis/data/raw/user_behavior_processed.csv
文件大小： 469.46 MB


In [3]:
sample_df = pd.read_csv(
    DATA_PATH,
    nrows=10000
)

print("样本数据形状：", sample_df.shape)
display(sample_df.head())

样本数据形状： (10000, 5)


,time,user_id,item_id,item_category,behavior_type
0,2025-12-06 02,98047837,232431562,4245,1
1,2025-12-09 20,97726136,383583590,5894,1
2,2025-12-18 11,98607707,64749712,2883,1
3,2025-12-06 10,98662432,320593836,6562,1
4,2025-12-16 21,98145908,290208520,13926,1


In [4]:
print("字段名称：")
print(sample_df.columns.tolist())

print("\n各字段的数据类型：")
print(sample_df.dtypes)

print("\n数据基本信息：")
sample_df.info()

字段名称：
['time', 'user_id', 'item_id', 'item_category', 'behavior_type']

各字段的数据类型：
time               str
user_id          int64
item_id          int64
item_category    int64
behavior_type    int64
dtype: object

数据基本信息：
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   time           10000 non-null  str  
 1   user_id        10000 non-null  int64
 2   item_id        10000 non-null  int64
 3   item_category  10000 non-null  int64
 4   behavior_type  10000 non-null  int64
dtypes: int64(4), str(1)
memory usage: 517.7 KB


In [5]:
print("各字段缺失值数量：")
print(sample_df.isna().sum())

print("\n完全重复行数量：")
print(sample_df.duplicated().sum())

print("\n行为类型分布：")
print(sample_df["behavior_type"].value_counts().sort_index())

各字段缺失值数量：
time             0
user_id          0
item_id          0
item_category    0
behavior_type    0
dtype: int64

完全重复行数量：
717

行为类型分布：
behavior_type
1    9430
2     195
3     276
4      99
Name: count, dtype: int64


In [6]:
sample_df["time"] = pd.to_datetime(
    sample_df["time"],
    format="%Y-%m-%d %H",
    errors="coerce"
)

print("转换后的字段类型：")
print(sample_df.dtypes)

print("\n无法转换的时间数量：")
print(sample_df["time"].isna().sum())

print("\n最早时间：", sample_df["time"].min())
print("最晚时间：", sample_df["time"].max())

转换后的字段类型：
time             datetime64[us]
user_id                   int64
item_id                   int64
item_category             int64
behavior_type             int64
dtype: object

无法转换的时间数量：
0

最早时间： 2025-11-18 00:00:00
最晚时间： 2025-12-18 23:00:00


In [7]:
duplicate_rows = sample_df[
    sample_df.duplicated(keep=False)
].sort_values(
    ["user_id", "item_id", "time", "behavior_type"]
)

print("重复记录数量：", len(duplicate_rows))
display(duplicate_rows.head(20))

重复记录数量： 1395


,time,user_id,item_id,item_category,behavior_type
9845,2025-12-03 11:00:00,2967122,17123536,11178,1
9854,2025-12-03 11:00:00,2967122,17123536,11178,1
9467,2025-12-03 16:00:00,2967122,176322914,2866,1
9503,2025-12-03 16:00:00,2967122,176322914,2866,1
9431,2025-11-22 05:00:00,2967122,272639944,5027,1
9476,2025-11-22 05:00:00,2967122,272639944,5027,1
9611,2025-11-21 15:00:00,2967122,273419428,169,1
9647,2025-11-21 15:00:00,2967122,273419428,169,1
2530,2025-11-29 21:00:00,10131505,66223566,4677,1
2908,2025-11-29 21:00:00,10131505,66223566,4677,1


In [8]:
behavior_mapping = {
    1: "浏览",
    2: "收藏",
    3: "加购",
    4: "购买"
}

behavior_summary = (
    sample_df["behavior_type"]
    .value_counts()
    .sort_index()
    .rename_axis("behavior_type")
    .reset_index(name="count")
)

behavior_summary["behavior_name"] = behavior_summary["behavior_type"].map(
    behavior_mapping
)

behavior_summary["percentage"] = (
    behavior_summary["count"] / len(sample_df) * 100
).round(2)

display(behavior_summary)

,behavior_type,count,behavior_name,percentage
0,1,9430,浏览,94.30
1,2,195,收藏,1.95
2,3,276,加购,2.76
3,4,99,购买,0.99


In [9]:
dtype_mapping = {
    "user_id": "int64",
    "item_id": "int64",
    "item_category": "int32",
    "behavior_type": "int8"
}

chunk_size = 500_000

chunk_reader = pd.read_csv(
    DATA_PATH,
    dtype=dtype_mapping,
    parse_dates=["time"],
    chunksize=chunk_size
)

print("分块读取器创建成功")



分块读取器创建成功


In [10]:
total_rows = 0
missing_counts = None
behavior_counts = {}
min_time = None
max_time = None
chunk_count = 0

for chunk in pd.read_csv(
    DATA_PATH,
    dtype=dtype_mapping,
    parse_dates=["time"],
    chunksize=chunk_size
):
    chunk_count += 1
    total_rows += len(chunk)

    # 缺失值统计
    current_missing = chunk.isna().sum()

    if missing_counts is None:
        missing_counts = current_missing
    else:
        missing_counts = missing_counts.add(
            current_missing,
            fill_value=0
        )

    # 行为类型统计
    current_behavior = chunk["behavior_type"].value_counts()

    for behavior_type, count in current_behavior.items():
        behavior_counts[behavior_type] = (
            behavior_counts.get(behavior_type, 0) + count
        )

    # 时间范围
    current_min_time = chunk["time"].min()
    current_max_time = chunk["time"].max()

    if min_time is None or current_min_time < min_time:
        min_time = current_min_time

    if max_time is None or current_max_time > max_time:
        max_time = current_max_time

    print(f"已完成第 {chunk_count} 个数据块，累计 {total_rows:,} 行")

    

已完成第 1 个数据块，累计 500,000 行
已完成第 2 个数据块，累计 1,000,000 行
已完成第 3 个数据块，累计 1,500,000 行
已完成第 4 个数据块，累计 2,000,000 行
已完成第 5 个数据块，累计 2,500,000 行
已完成第 6 个数据块，累计 3,000,000 行
已完成第 7 个数据块，累计 3,500,000 行
已完成第 8 个数据块，累计 4,000,000 行
已完成第 9 个数据块，累计 4,500,000 行
已完成第 10 个数据块，累计 5,000,000 行
已完成第 11 个数据块，累计 5,500,000 行
已完成第 12 个数据块，累计 6,000,000 行
已完成第 13 个数据块，累计 6,500,000 行
已完成第 14 个数据块，累计 7,000,000 行
已完成第 15 个数据块，累计 7,500,000 行
已完成第 16 个数据块，累计 8,000,000 行
已完成第 17 个数据块，累计 8,500,000 行
已完成第 18 个数据块，累计 9,000,000 行
已完成第 19 个数据块，累计 9,500,000 行
已完成第 20 个数据块，累计 10,000,000 行
已完成第 21 个数据块，累计 10,500,000 行
已完成第 22 个数据块，累计 11,000,000 行
已完成第 23 个数据块，累计 11,500,000 行
已完成第 24 个数据块，累计 12,000,000 行
已完成第 25 个数据块，累计 12,256,906 行


In [11]:
print("完整数据总行数：", f"{total_rows:,}")
print("数据块数量：", chunk_count)

print("\n各字段缺失值：")
print(missing_counts.astype("int64"))

print("\n行为类型数量：")
for behavior_type in sorted(behavior_counts):
    print(
        behavior_type,
        behavior_mapping[behavior_type],
        f"{behavior_counts[behavior_type]:,}"
    )

print("\n最早时间：", min_time)
print("最晚时间：", max_time)

完整数据总行数： 12,256,906
数据块数量： 25

各字段缺失值：
time             0
user_id          0
item_id          0
item_category    0
behavior_type    0
dtype: int64

行为类型数量：
1 浏览 11,550,581
2 收藏 242,556
3 加购 343,564
4 购买 120,205

最早时间： 2025-11-18 00:00:00
最晚时间： 2025-12-18 23:00:00


In [12]:
full_behavior_summary = pd.DataFrame(
    {
        "behavior_type": list(behavior_counts.keys()),
        "count": list(behavior_counts.values())
    }
).sort_values("behavior_type")

full_behavior_summary["behavior_name"] = (
    full_behavior_summary["behavior_type"]
    .map(behavior_mapping)
)

full_behavior_summary["percentage"] = (
    full_behavior_summary["count"]
    / total_rows
    * 100
).round(2)

display(full_behavior_summary)


,behavior_type,count,behavior_name,percentage
0,1,11550581,浏览,94.24
2,2,242556,收藏,1.98
1,3,343564,加购,2.80
3,4,120205,购买,0.98
